# Meta Model


Model family: **market**. News event keys, labels, partitions, and notebook workflow are shared.

## Purged Validation

- **Purpose:** Apply `purged_validation.py`'s `PurgedKFold` directly to the development partition.
- **Settings:** `cv=5` contiguous folds; `pct_embargo=0.01`; `random_state=42`; labels = {0, 1}; sealed chronological holdout.
- **Data:** Use development meta features, action labels, sample weights, and event information intervals.
- **Findings:**
- **Decision:** Purge overlapping intervals, exclude holdout from every fold, and maximize weighted F1 with weighted log loss used for ties.

In [ ]:
from pathlib import Path

from IPython import get_ipython

get_ipython().run_line_magic("matplotlib", "inline")

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[2]

from src.modeling.purged_validation import index_events
from src.modeling.purged_validation import PurgedKFold
from src.modeling.model_workflow import (
    build_model_evaluation_table,
    build_meta_model_frame,
    compute_stage_importance,
    finalize_meta_model,
    get_meta_feature_columns,
    plot_feature_importance,
    plot_learning_curves,
    plot_model_evaluation,
    run_model_selection_workflow,
    score_binary_predictions,
)
from src.preprocessing.market_data import ResearchPaths

RANDOM_STATE = 42
CV_SPLITS = 5
PCT_EMBARGO = 0.01
TRAIN_SIZES = [0.20, 0.40, 0.60, 0.80, 1.00]
PERIOD = "2025-02-01_2025-12-31"
CANDIDATE_SETTINGS = {
    "boosting": {"n_estimators": 100, "learning_rate": 0.10, "max_depth": 1},
    "bagging": {"n_estimators": 120, "max_samples": 0.80,
                "max_features": 1.0, "min_weight_fraction_leaf": 0.0},
    "random_forest": {"n_estimators": 120, "max_features": "sqrt",
                      "min_weight_fraction_leaf": 0.0},
}
PARAMETER_GRIDS = {
    "boosting": {"model__learning_rate": [0.03, 0.10, 0.30]},
    "bagging": {"model__max_samples": [0.60, 0.80, 1.00]},
    "random_forest": {"model__max_features": ["sqrt", 0.50, 1.00]},
}
paths = ResearchPaths(PROJECT_ROOT, model_kind="market", period=PERIOD)
event_path = paths.event("model")
artifact_dir = PROJECT_ROOT / "data/model_artifact/market"
artifact_dir.mkdir(parents=True, exist_ok=True)

events = index_events(pd.read_parquet(event_path))
split_manifest = pd.read_parquet(artifact_dir / "split_manifest.parquet")
primary_predictions = index_events(pd.read_parquet(artifact_dir / "primary_predictions.parquet"))

development_primary = primary_predictions[
    primary_predictions["partition"].eq("development")
]
manifest_development = pd.MultiIndex.from_frame(split_manifest.loc[
    split_manifest["partition"].eq("development"), ["symbol", "event_start"]
])
if set(development_primary.index) != set(manifest_development):
    raise ValueError("Primary OOF rows must match the fixed development partition")

development_events = events.loc[development_primary.index]
primary_oof_contract = development_primary.rename(columns={
    "primary_side": "prediction",
    "primary_probability": "probability",
})
meta_development = build_meta_model_frame(
    development_events,
    primary_oof_contract[["prediction", "probability", "prediction_source"]],
)
meta_features = get_meta_feature_columns(meta_development, model_kind="market")

X_development = meta_development[meta_features]
y_development = meta_development["meta_label"].astype("int8")
w_development = meta_development["sample_weight"].astype("float64")
t1_development = meta_development["event_end"]
CLASS_LABELS = [0, 1]
STAGE_SCORING = "f1"

cv = PurgedKFold(CV_SPLITS, t1=t1_development, pct_embargo=PCT_EMBARGO)
fold_rows = []
for fold, (train, test) in enumerate(cv.split(X_development)):
    fold_rows.append({
        "fold": fold,
        "train_events": len(train),
        "test_events": len(test),
        "test_start": t1_development.index.get_level_values("event_start")[test].min(),
        "test_end": t1_development.iloc[test].max(),
    })

display(pd.DataFrame(fold_rows).set_index("fold"))
display(pd.Series({
    "development_events": len(meta_development),
    "act_labels": int(y_development.sum()),
    "pass_labels": int((1 - y_development).sum()),
    "features": len(meta_features),
}, name="value").to_frame())

## Ensemble Methods

- **Purpose:** Compare the boosting, bagging, and random-forest factories from `ensemble_methods.py`.
- **Settings:** Entropy trees; boosting: 100 stumps, rate 0.10; bagging: 120 bootstrapped trees, samples 0.80, balanced classes; RF: 120 bootstrapped trees, sqrt features, balanced_subsample; seed 42; `n_jobs=1`.
- **Data:** Generate comparable purged OOF predictions for all three candidates from the development partition.
- **Findings:**
- **Decision:** Rank candidates by maximum weighted F1 and use weighted log loss for ties.

In [ ]:
workflow = run_model_selection_workflow(
    X_development,
    y_development,
    w_development,
    t1_development,
    scoring=STAGE_SCORING,
    cv=CV_SPLITS,
    pct_embargo=PCT_EMBARGO,
    random_state=RANDOM_STATE,
    n_jobs=1,
    candidate_settings=CANDIDATE_SETTINGS, parameter_grids=PARAMETER_GRIDS,
)
candidates = workflow.estimators
candidate_oof = workflow.oof_predictions
comparison = workflow.comparison
selected_name = workflow.selected_name
selected_estimator = workflow.selected_estimator
display(comparison.rename(index=lambda name: name.replace("_", " ").title()))
print(f"Selected baseline family: {selected_name}")

## Ensemble Methods Learning Curves

- **Purpose:** Use `get_weighted_learning_curve` to compare how the three baseline candidates learn across purged folds.
- **Settings:** `train_sizes=(0.20,0.40,0.60,0.80,1.00)`; stratified subsets; `cv=5`; embargo 1%; seed 42; error = 1 − weighted F1, mean ± standard error.
- **Data:** Build learning curves from development folds only.
- **Findings:**
- **Decision:** Interpret the train-validation gap as a diagnostic rather than a model-selection result.

In [ ]:
ensemble_learning_curves = plot_learning_curves(
    candidates,
    "Ensemble Methods Learning Curves",
    features=X_development,
    labels=y_development,
    sample_weight=w_development,
    cv=cv,
    train_sizes=TRAIN_SIZES,
    class_labels=CLASS_LABELS,
    positive_label=1,
    scoring=STAGE_SCORING,
    random_state=RANDOM_STATE,
)

## Ensemble Methods Feature Importance

- **Purpose:** Apply estimator-aware MDI, MDA, and SFI to the selected baseline family.
- **Settings:** `cv=5`; `pct_embargo=0.01`; `random_state=42`; `top_k=15`; MDA and SFI use weighted F1.
- **Data:** Calculate importance for all 51 meta features: 49 primary features plus primary side and confidence.
- **Findings:**
- **Decision:** Treat importance as a diagnostic rather than causal evidence or a new selection step.

In [ ]:
ensemble_importance_results, ensemble_importance_scores = compute_stage_importance(
    selected_estimator,
    features=X_development,
    labels=y_development,
    sample_weight=w_development,
    information_sets=t1_development,
    scoring=STAGE_SCORING,
    cv=CV_SPLITS,
    pct_embargo=PCT_EMBARGO,
    random_state=RANDOM_STATE,
)
plot_feature_importance(
    ensemble_importance_results,
    f"Ensemble Methods Feature Importance: {selected_name}",
)
display(ensemble_importance_scores.to_frame())

## Ensemble Methods Model Evaluation

- **Purpose:** Compare baseline weighted accuracy, precision, recall, F1, log loss, confusion matrices, PR curves, and ROC curves.
- **Settings:** Labels = {0, 1}; positive label = 1; sample-weighted metrics, confusion matrices, PR/AP and ROC/AUC.
- **Data:** Evaluate the development OOF predictions for all three candidates.
- **Findings:**
- **Decision:** Do not use holdout curves to adjust thresholds.

In [ ]:
display(build_model_evaluation_table(
    candidate_oof,
    y_development,
    w_development,
    class_labels=CLASS_LABELS,
    positive_label=1,
))

plot_model_evaluation(
    candidate_oof,
    y_development,
    w_development,
    "Ensemble Methods Model Evaluation",
    class_labels=CLASS_LABELS,
    positive_label=1,
)

## Hyperparameter Tuning

- **Purpose:** Use the shared model-selection workflow to tune only the family selected by the baseline comparison.
- **Settings:** `cv=5`; embargo 1%; seed 42; `n_jobs=1`; selected-family grid: boosting rate [0.03,0.10,0.30], bagging samples [0.60,0.80,1.00], RF features [sqrt,0.50,1.00].
- **Data:** Apply sample weights within purged development folds for the selected family.
- **Findings:**
- **Decision:** Select the best mean weighted CV F1; exact ties follow grid order; exclude holdout feedback.

In [ ]:
final_name = workflow.selected_name
final_configuration = workflow.final_configuration
final_estimator = workflow.final_estimator
final_oof = workflow.final_oof
tuning = workflow.tuning
tuned_estimators = {f"{final_name}_tuned": final_estimator}
display(tuning)
print(f"Selected tuned family: {final_name}")

## Hyperparameter Tuning Learning Curves

- **Purpose:** Diagnose the selected tuned estimator with the same weighted learning-curve helper.
- **Settings:** `train_sizes=(0.20,0.40,0.60,0.80,1.00)`; stratified subsets; `cv=5`; embargo 1%; seed 42; error = 1 − weighted F1, mean ± standard error.
- **Data:** Build the tuned estimator's learning curve from development folds only.
- **Findings:**
- **Decision:** Do not revise the model or grid from holdout learning behavior.

In [ ]:
tuned_learning_curves = plot_learning_curves(
    tuned_estimators,
    "Hyperparameter Tuning Learning Curves",
    features=X_development,
    labels=y_development,
    sample_weight=w_development,
    cv=cv,
    train_sizes=TRAIN_SIZES,
    class_labels=CLASS_LABELS,
    positive_label=1,
    scoring=STAGE_SCORING,
    random_state=RANDOM_STATE,
)

## Hyperparameter Tuning Feature Importance

- **Purpose:** Apply the same estimator-aware MDI, MDA, and SFI methods to the final tuned family.
- **Settings:** `cv=5`; `pct_embargo=0.01`; `random_state=42`; `top_k=15`; MDA and SFI use weighted F1.
- **Data:** Calculate importance for all 51 meta features: 49 primary features plus primary side and confidence.
- **Findings:**
- **Decision:** Treat changes from baseline as diagnostics rather than feature selection.

In [ ]:
tuned_importance_results, tuned_importance_scores = compute_stage_importance(
    final_estimator,
    features=X_development,
    labels=y_development,
    sample_weight=w_development,
    information_sets=t1_development,
    scoring=STAGE_SCORING,
    cv=CV_SPLITS,
    pct_embargo=PCT_EMBARGO,
    random_state=RANDOM_STATE,
)
plot_feature_importance(
    tuned_importance_results,
    f"Hyperparameter Tuning Feature Importance: {final_name}",
)
importance = pd.DataFrame({
    "mdi": tuned_importance_results["MDI"]["mean"],
    "mda": tuned_importance_results["MDA"]["mean"],
    "sfi_neg_log_loss" if STAGE_SCORING == "neg_log_loss" else "sfi_f1": (
        tuned_importance_results["SFI"]["mean"]
    ),
}).sort_values("mda", ascending=False)
display(importance.head(15))
display(tuned_importance_scores.to_frame())

## Hyperparameter Tuning Model Evaluation

- **Purpose:** Compare the selected family's baseline and tuned accuracy, precision, recall, F1, log loss, and diagnostic curves before final evaluation.
- **Settings:** Labels = {0, 1}; positive label = 1; sample-weighted metrics, confusion matrices, PR/AP and ROC/AUC. One final holdout evaluation per run.
- **Data:** Use development OOF results and one final holdout evaluation, then persist the model and prediction provenance.
- **Findings:**
- **Decision:** Fix the configuration from development OOF and evaluate holdout exactly once.

In [ ]:
tuning_evaluation_oof = {
    f"{selected_name}_baseline": candidate_oof[selected_name],
    f"{final_name}_tuned": final_oof,
}
display(build_model_evaluation_table(
    tuning_evaluation_oof,
    y_development,
    w_development,
    class_labels=CLASS_LABELS,
    positive_label=1,
))
plot_model_evaluation(
    tuning_evaluation_oof,
    y_development,
    w_development,
    "Hyperparameter Tuning Model Evaluation",
    class_labels=CLASS_LABELS,
    positive_label=1,
)

meta_predictions = finalize_meta_model(
    meta_development,
    events,
    primary_predictions,
    estimator=final_estimator,
    final_oof=final_oof,
    feature_columns=meta_features,
    artifact_dir=artifact_dir,
    selected_candidate=final_name,
    best_configuration=final_configuration,
    random_state=RANDOM_STATE,
)
holdout_events = meta_predictions.loc[meta_predictions["partition"].eq("holdout")]
holdout_action = holdout_events["meta_action"]
holdout_probability = holdout_events["meta_probability"]

holdout_prediction_frame = pd.DataFrame({
    "prediction": holdout_action,
    "probability": holdout_probability,
}, index=holdout_events.index)
holdout_scores = score_binary_predictions(
    holdout_events["meta_label"],
    holdout_prediction_frame["prediction"],
    holdout_prediction_frame["probability"],
    holdout_events["sample_weight"],
    class_labels=CLASS_LABELS,
    positive_label=1,
)
holdout_metrics = pd.Series({
    "accuracy": holdout_scores["accuracy"],
    "precision": holdout_scores["precision"],
    "recall": holdout_scores["recall"],
    "f1": holdout_scores["f1"],
    "log_loss": holdout_scores["log_loss"],
}, name="holdout")
display(holdout_metrics.to_frame())
display(build_model_evaluation_table(
    {"final_holdout": holdout_prediction_frame},
    holdout_events["meta_label"],
    holdout_events["sample_weight"],
    class_labels=CLASS_LABELS,
    positive_label=1,
))
plot_model_evaluation(
    {"final_holdout": holdout_prediction_frame},
    holdout_events["meta_label"],
    holdout_events["sample_weight"],
    "Final One-Time Holdout Evaluation",
    class_labels=CLASS_LABELS,
    positive_label=1,
)

print(artifact_dir / "meta_model.joblib")